In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from crewai import Agent, Task, Crew, Process, LLM

import os
import asyncio
import nest_asyncio

nest_asyncio.apply()

model = os.getenv('MODEL_NAME', 'gpt-4o')
api_base = os.getenv('OPENAI_API_BASE', '')

def build_email_crew(context: str, tone: str, recipient: str) -> str:
    llm = LLM(model=model, temperature=0.3, api_base=api_base)

    analyst = Agent(
        role="邮件上下文分析师",
        goal="理解邮件语境，提取关键信息，并定义邮件结构",
        backstory="你是一位专业的商务沟通分析师，擅长将复杂情况提炼为清晰的邮件需求。",
        llm=llm,
        verbose=True,
    )

    writer = Agent(
        role="专业邮件撰写人",
        goal="撰写清晰、简洁、高效的专业邮件",
        backstory="你是一位专业文案撰稿人，专注于撰写能获得回复的商务邮件。",
        llm=llm,
        verbose=True,
    )

    analyze_task = Task(
        description=f"""分析以下邮件需求：
        语境：{context}
        收件人：{recipient}
        期望语气：{tone}

        请提取：目的、要点、行动号召（CTA）、以及建议的邮件主题。""",
        agent=analyst,
        expected_output="结构化的邮件简报：目的、要点、行动号召和建议的邮件主题",
    )

    write_task = Task(
        description=f"""根据分析结果，撰写一封完整的商务邮件。
        语气：{tone}。收件人：{recipient}。
        请包含：邮件主题、称呼、正文段落、结尾敬语、签名占位符。
        正文请保持简洁——不超过200字。""",
        agent=writer,
        expected_output="可直接发送的完整格式邮件",
        context=[analyze_task],
    )

    crew = Crew(
        agents=[analyst, writer],
        tasks=[analyze_task, write_task],
        process=Process.sequential,
        verbose=True,
        tracing=True
    )

    result = asyncio.run(crew.kickoff_async())
    return str(result)

In [3]:
context = '跟进上周二的产品演示。他们表现出兴趣但尚未回复。'
tone = '专业且友好'
recipient = '潜在客户'

result = build_email_crew(context=context, tone=tone, recipient=recipient)

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 41bfdbfe-daf9-4192-88a0-3a1c5888216a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 分析以下邮件需求：                                                                                       │
│          语境：跟进上周二的产品演示。他们表现出兴趣但尚未回复。                                                 │
│          收件人：潜在客户                                                                                       │
│          期望语气：专业且友好                                                                                   │
│                                                                                                                 │
│          请提取：目的、要点、行动号召（CTA）、以及建议的邮件主题。                                              │
│  ID: 2bd7004f-b512-4362-ba3b-8271691e5d2f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 邮件上下文分析师                                                                                        │
│                                                                                                                 │
│  Task: 分析以下邮件需求：                                                                                       │
│          语境：跟进上周二的产品演示。他们表现出兴趣但尚未回复。                                                 │
│          收件人：潜在客户                                                                                       │
│          期望语气：专业且友好                                                                                   │
│                                                                                                                 │
│          请提取：目的、要点、行动号召（CTA）、以及建议的邮件主题。                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 邮件上下文分析师                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **结构化的邮件简报**                                                                                           │
│                                                                                                                 │
│  - **目的：**                                                                                                   │
│  跟进上周二的产品演示，巩固潜在客户已表现出的兴趣信号，化解潜在的决策迟疑，并将单向展示转化为双向业务对话，推   │
│  动销售流程进入需求对齐或方案评估阶段。                                                                         │
│                                                                                                                 │
│  - **要点：**                                                                                                   │
│    1. **精准回溯：** 开篇致谢并具体提及周二演示中客户重点关注的模块或痛点，证明团队认真聆听了其反馈。           │
│    2. **价值增量：**                                                                                            │
│  为避免“单纯催促”，随信补充1-2项高相关度资料（如：同行业落地案例、演示中承诺的技术参数详解、或初步的ROI测算框   │
│  架），体现专业支持能力。                                                                                       │
│    3. **节奏把控：** 语气保持专业、真诚且不具压迫感，明确认可对方内部评估需要时间，展现对业务节奏的尊重。       │
│    4. **灵活接口：** 预留低门槛的沟通路径，表明团队可根据客户实际工作安排随时调整对接方式。                     │
│                                                                                                                 │
│  - **行动号召（CTA）：**                                                                                        │
│  邀请客户预约一次15-20分钟的简短专项通话，聚焦其核心业务目标与下一步实施规划；或请客户直接回复本邮件，提供本周  │
│  /下周方便的沟通时间段及首选联系方式。                                                                          │
│                                                                                                                 │
│  - **建议的邮件主题：** 跟进：周二产品演示后续事宜 & 下一步计划探讨 / 关于周二演示的进一步沟通 /                │
│  感谢参与周二演示，期待与您继续交流                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 专业邮件撰写人                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  邮件主题：感谢参与周二演示，期待与您进一步沟通                                                                 │
│                                                                                                                 │
│  称呼：[客户姓名]您好，                                                                                         │
│                                                                                                                 │
│  正文：感谢您上周二拨冗参与产品演示。针对您重点关注的[具体模块/痛点]，随信附上同行业落地案例与初步ROI测算框架   │
│  ，供贵司内部评估参考。充分理解决策需要周期，我们将完全尊重并配合您的业务节奏。若您方便，诚邀预约15-20分钟专项  │
│  通话，聚焦核心目标与下一步规划；或直接回复本邮件告知本周/下周方便的时间段。静候佳音。                          │
│                                                                                                                 │
│  结尾敬语：顺祝商祺，                                                                                           │
│                                                                                                                 │
│  签名占位符：[您的姓名] | [您的职位]                                                                            │
│  [公司名称]                                                                                                     │
│  [联系电话] | [邮箱地址]                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 根据分析结果，撰写一封完整的商务邮件。                                                                   │
│          语气：专业且友好。收件人：潜在客户。                                                                   │
│          请包含：邮件主题、称呼、正文段落、结尾敬语、签名占位符。                                               │
│          正文请保持简洁——不超过200字。                                                                          │
│  Agent: 专业邮件撰写人                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 41bfdbfe-daf9-4192-88a0-3a1c5888216a                                                                       │
│  Final Output: 邮件主题：感谢参与周二演示，期待与您进一步沟通                                                   │
│                                                                                                                 │
│  称呼：[客户姓名]您好，                                                                                         │
│                                                                                                                 │
│  正文：感谢您上周二拨冗参与产品演示。针对您重点关注的[具体模块/痛点]，随信附上同行业落地案例与初步ROI测算框架   │
│  ，供贵司内部评估参考。充分理解决策需要周期，我们将完全尊重并配合您的业务节奏。若您方便，诚邀预约15-20分钟专项  │
│  通话，聚焦核心目标与下一步规划；或直接回复本邮件告知本周/下周方便的时间段。静候佳音。                          │
│                                                                                                                 │
│  结尾敬语：顺祝商祺，                                                                                           │
│                                                                                                                 │
│  签名占位符：[您的姓名] | [您的职位]                                                                            │
│  [公司名称]                                                                                                     │
│  [联系电话] | [邮箱地址]                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

邮件主题：感谢参与周二演示，期待与您进一步沟通

称呼：[客户姓名]您好，

正文：感谢您上周二拨冗参与产品演示。针对您重点关注的[具体模块/痛点]，随信附上同行业落地案例与初步ROI测算框架，供贵司内部评估参考。充分理解决策需要周期，我们将完全尊重并配合您的业务节奏。若您方便，诚邀预约15-20分钟专项通话，聚焦核心目标与下一步规划；或直接回复本邮件告知本周/下周方便的时间段。静候佳音。

结尾敬语：顺祝商祺，

签名占位符：[您的姓名] | [您的职位]
[公司名称]
[联系电话] | [邮箱地址]


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

model = os.getenv('MODEL_NAME', 'gpt-4o')
llm = ChatOpenAI(model=model, temperature=0.3, extra_body={"enable_thinking": False})

class EmailState(TypedDict):
    context: str
    tone: str
    recipient: str
    analysis: str
    email: str

def analyze_node(state: EmailState) -> EmailState:
    messages = [
        SystemMessage(content="你是一位专业的商务沟通分析师。"),
        HumanMessage(content=f"""分析以下邮件需求：
        语境: {state['context']}
        收件人: {state['recipient']}
        期望语气: {state['tone']}
        请提取: 目的、要点、CTA、建议主题"""),
    ]
    result = llm.invoke(messages)
    return {
        "context": state["context"],
        "tone": state["tone"],
        "recipient": state["recipient"],
        "analysis": result.content,
        "email": "",
    }

def write_node(state: EmailState) -> EmailState:
    messages = [
        SystemMessage(content="你是一位专业的商务邮件撰稿人。"),
        HumanMessage(content=f"""根据分析结果撰写一封完整的商务邮件：
        分析结果: {state['analysis']}
        语气: {state['tone']}
        收件人: {state['recipient']}
        请包含: 邮件主题、称呼、正文段落、结尾敬语、签名占位符
        正文不超过200字"""),
    ]
    result = llm.invoke(messages)
    return {
        "context": state["context"],
        "tone": state["tone"],
        "recipient": state["recipient"],
        "analysis": state["analysis"],
        "email": result.content,
    }

# 构建图
workflow = StateGraph(EmailState)
workflow.add_node("analyze", analyze_node)
workflow.add_node("write", write_node)
workflow.set_entry_point("analyze")
workflow.add_edge("analyze", "write")
workflow.add_edge("write", END)
app = workflow.compile()


input_state: EmailState = {
    "context": context,
    "tone": tone,
    "recipient": recipient,
    "analysis": "",
    "email": ""
}
# 执行
for chunk in app.stream(input=input_state, stream_mode="messages"):
    msg, meta = chunk
    if hasattr(msg, 'content'):
        print(getattr(msg, 'content'), end='', flush=False)


基于您提供的语境和要求，以下是针对这封跟进邮件的详细分析与建议内容：

### 1. 目的 (Purpose)
*   **重新建立联系**：在潜在客户表现出兴趣但沉默后，温和地提醒对方之前的互动，避免让对话完全冷却。
*   **消除疑虑/提供价值**：通过再次强调演示中的核心价值或提供额外信息，降低客户的决策门槛。
*   **推动进程**：将“未回复”的状态转化为具体的下一步行动（如简短通话、发送案例研究或安排二次会议）。

### 2. 要点 (Key Points)
*   **回顾与确认**：提及上周二的产品演示，并感谢对方的时间。简要重申他们在演示中表现出的具体兴趣点（如果记得的话，这会显得更个性化）。
*   **价值重申**：用一两句话概括该产品如何解决他们的核心痛点，或者提及演示中他们最关注的那个功能/优势。
*   **低压力介入**：表明理解对方可能很忙，因此不需要立即做出决定，只是希望保持沟通渠道畅通。
*   **提供便利选项**：主动提出可以发送额外的资料（如案例研究、定价表或技术白皮书），以减少对方的搜索成本。

### 3. CTA (Call to Action / 行动号召)
*   **首选 CTA**：询问是否方便进行一个 10-15 分钟的简短电话，以解答任何遗留问题或讨论下一步。
    *   *示例话术*：“如果您有任何疑问，或者想进一步探讨某个功能，我们是否可以安排一个 10 分钟的简短通话？”
*   **备选 CTA**：如果对方表示暂时没空，请求许可发送一份相关的成功案例或详细资料供其参考。
    *   *示例话术*：“如果您目前较忙，我也可以先发送一份针对您行业的案例研究供您参考，您看如何？”

### 4. 建议主题 (Suggested Subject Lines)
为了保持专业且友好，同时提高打开率，建议使用以下主题之一：

*   **直接且友好型**：Follow-up: Our discussion on Tuesday / 跟进：我们周二的讨论
*   **价值导向型**：Thoughts on [Product Name] for [Client's Company]? / 关于 [产品名称] 用于 [客户公司] 的一些想法
*   **轻量级/低压型**：Quick check-in 